# NIG-TimeNet (PyTorch): the autoregressive candle stream

**Runtime > Run all.** One notebook from raw candles to a direction-forecasting model that works the way a text
Transformer does: the window is `CONTEXT_LEN + FUTURE_LEN` candles (48 by default = 32 context + 16 future),
the model predicts the next candle, its own prediction is appended as the next input, and that repeats
candle by candle. The horizon is a free parameter: 4, 8, 16 or any other number works unchanged, and the
EVALUATION horizon can even differ from the TRAINING horizon. Coins are free too: all of them, or an
explicit list.

Training follows the same shape (updated for it): every horizon step is predicted candle by candle, with
scheduled sampling - with a rising probability the appended candle is the model's OWN prediction, detached, so
the model learns to consume its own outputs without the noise in them becoming a learning signal. The trainer is
deliberate about unpredictable moves and the cowardice they teach, and the loss is selectable (`LOSS` below):
the default "czar" scores each return in units of its window's LOCAL volatility (a shock is bounded by the
Cauchy kernel, a shrunk-to-zero prediction on a real move stays expensive, flat noise cannot pay), while
"huber" keeps the v1 robust loss (Huber cap + adaptive outlier weight). The DIRECTION term keeps its full
weight in both. Selection is the validation direction accuracy of the full rollout, next to the persistence
and always-up trivial baselines and a shuffled-labels control at its honest null.

In [ ]:
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "feat/ar-stream"
EXPECT_DEVICE = "cuda"                         # "cpu" only for a deliberate CPU run; a cuda run on a CPU VM fails
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # clones' scratch, logs, git credentials helper: never /tmp (rule E2)
TORCH_DIR = "/content/nig-timenet-torch"
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
EXTRA_MODULES = ()                             # torch/pandas/numpy ship with Colab
SOURCE = "drive"                               # "drive" (your Drive candle files), "binance" (live futures API, synthetic fallback) or "synthetic"
COINS = "all"                                  # 'all', 'all_global' or an explicit tuple: ("BTCUSDT", "ETHUSDT")
DAYS = 120                                     # days of hourly candles per coin (the newest DAYS days are kept)
INTERVAL = "1h"
RAW_ROOT = "/content/drive/MyDrive"              # the Drive folder holding history_1h/<SYMBOL>.csv (crypto-signal-prediction layout)
CONTEXT_LEN = 32                               # the base window: 32 candles the model conditions on
HORIZON_LEN = 16                               # candles the model GENERATES while training: 4, 8, 16, ... any number
EVAL_HORIZON = 8                               # candles to SCORE in the rollout: free, independent of HORIZON_LEN
LOSS = "czar"                                   # "czar" (anti-cowardice, shock-bounded) or "huber" (v1)
CHANGES = {}                                   # ArStreamSettings overrides, e.g. {"epochs": 20, "ss_end": 0.8}
VAL_SHARE, TEST_SHARE = 0.15, 0.15

## 1. Setup
The first cell is plain code: nothing of the package exists on the VM yet, so it downloads the one stdlib-only file
`colab/bootstrap.py` with the token and runs it. Everything after it is a normal import.

In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))

In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR)})
install = boot["install_package"](TORCH_DIR, modules=EXTRA_MODULES)

In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)

## 2. Data: raw candles -> gap-free windows
Raw OHLCV bars are loaded as-is (no re-aggregation: every window is an independent sample; SOURCE="drive"
reads the candle files your Drive already holds - the crypto-signal-prediction layout `history_1h/<SYMBOL>.csv`) and turned into five
scale-free per-candle features (return, two wick ratios, body position, volume ratio). Windows are cut only
inside gap-free stretches, so no window ever spans a hole in the exchange data. The windows carry
`FUTURE_LEN = max(HORIZON_LEN, EVAL_HORIZON)` future candles so the same data serves a training horizon and a
DIFFERENT evaluation horizon.

In [ ]:
from nig_timenet_torch.data.ar_windows import raw_candles
FUTURE_LEN = max(HORIZON_LEN, EVAL_HORIZON)
frames = raw_candles(SOURCE, COINS, DAYS, INTERVAL, root=RAW_ROOT, seed=0)
print(len(frames), "coins,", {k: len(v) for k, v in list(frames.items())[:3]}, "bars ...")

In [ ]:
from nig_timenet_torch.data.ar_windows import build_windows, candle_features, contiguous_runs, fit_volume_median
coins = {}
for name, df in frames.items():
    feats = candle_features(df, fit_volume_median(df))
    runs = contiguous_runs(df.index, INTERVAL)
    coins[name] = build_windows(feats, runs, CONTEXT_LEN, FUTURE_LEN)
    print(f"{name}: {coins[name].n} windows from {len(runs)} gap-free stretch(es)")

In [ ]:
from nig_timenet_torch.data.ar_windows import split_windows
data = split_windows(coins, VAL_SHARE, TEST_SHARE)
print({k: v.shape for k, v in data.items()})

## 3. Train - updated for the (context + horizon) shape
`fit_ar_stream` predicts every horizon step candle by candle (the deployment shape), mixes the model's own
detached predictions into the inputs on a rising schedule, scores the return with the selected LOSS kind ("czar" by default: local-volatility z units,
Cauchy-bounded shocks, anti-cowardice; or "huber": the v1 Huber cap + adaptive outlier weight), keeps the direction term at full weight, and selects the epoch by the
validation direction accuracy of the full autoregressive rollout.

In [ ]:
from nig_timenet_torch.trainer.ar_stream import ArStreamSettings, build_ar_stream_model, fit_ar_stream
settings = ArStreamSettings(context_len=CONTEXT_LEN, horizon_len=HORIZON_LEN, device=EXPECT_DEVICE, loss=LOSS, **CHANGES)
model = build_ar_stream_model(settings, n_features=data["train_X"].shape[2])
history = fit_ar_stream(model, data["train_X"], data["train_Y"], data["train_dirs"],
                        data["val_X"], data["val_Y"], data["val_dirs"], settings)
print("best epoch", history["best_epoch"], "val direction accuracy", round(history["best_val_dir_acc"], 4))

## 4. Autoregressive rollout: direction first, at a FREE horizon
The test is the honest one: the model eats only its own predictions for the whole rollout - `EVAL_HORIZON`
candles, which may differ from the horizon it was trained on (the position table extends itself). Direction
accuracy is reported overall and per step, always next to the two trivial baselines (persistence sign,
always-up): the model only counts when it beats them.

In [ ]:
from nig_timenet_torch.eval.ar_stream import rollout_metrics
report = rollout_metrics(model, data["test_X"], data["test_Y"], data["test_dirs"], data["test_last_ret"],
                         EXPECT_DEVICE, horizon_len=EVAL_HORIZON)
print("direction", round(report["direction_accuracy"], 4), "persistence", round(report["persistence_accuracy"], 4),
      "always-up", round(report["always_up_accuracy"], 4), "gap", round(report["trivial_gap"], 4))
print("per step:", [round(a, 3) for a in report["direction_accuracy_per_step"]])

## 5. Controls and self-checks
The shuffled-labels control must land at its honest null - the majority agreement of a biased model, ~50% for a balanced one:
anything clearly higher means the scoring leaks and any claim is void. The shape checks pin the window contract (32 + 16 by default, any other pair equally valid).

In [ ]:
import numpy as np
from nig_timenet_torch.eval.ar_stream import direction_control
control = direction_control(model, data["test_X"], data["test_dirs"], EXPECT_DEVICE, horizon_len=EVAL_HORIZON)
print("shuffled control", round(control["shuffled_direction_accuracy"], 4), "expected ~", round(control["expected"], 4))
assert data["train_X"].shape[1] == CONTEXT_LEN and data["train_Y"].shape[1] == FUTURE_LEN
assert 1 <= EVAL_HORIZON <= FUTURE_LEN and set(np.unique(data["train_dirs"])) <= {-1.0, 0.0, 1.0}
assert abs(control["shuffled_direction_accuracy"] - control["expected"]) < 0.1
print("self-checks passed")